GCP: ship the model and call the cat/dog model

Assumes gcloud is logged in the project ID ex: sbox-johnny-001, 

and local files:
models/image_classifier.h5

for serving you can also optionally include the dog_test.jpg / cat_test.jpg.

In [2]:
import os
import subprocess

PROJECT = "sbox-johnny-001"
REGION = "us-central1"
BUCKET = "gs://sbox-johnny-001-mldemo"
SERVICE = "cat-dog-classifier"
MODEL_LOCAL = "models/image_classifier.h5"

os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT

def sh(cmd: str):
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=True)

print("project:", PROJECT)

project: sbox-johnny-001


## 1. Upload model to GCS

In [3]:
# Create bucket once if missing, then upload the .h5 only
sh(
    f"gcloud storage buckets describe {BUCKET} >/dev/null 2>&1 "
    f"|| gcloud storage buckets create {BUCKET} --location={REGION} --uniform-bucket-level-access"
)
sh(f"gcloud storage cp {MODEL_LOCAL} {BUCKET}/models/image_classifier.h5")
sh(f"gcloud storage ls {BUCKET}/models/")

+ gcloud storage buckets describe gs://sbox-johnny-001-mldemo >/dev/null 2>&1 || gcloud storage buckets create gs://sbox-johnny-001-mldemo --location=us-central1 --uniform-bucket-level-access
+ gcloud storage cp models/image_classifier.h5 gs://sbox-johnny-001-mldemo/models/image_classifier.h5


Copying file://models/image_classifier.h5 to gs://sbox-johnny-001-mldemo/models/image_classifier.h5
  
.....................................................................................................

Average throughput: 5.4MiB/s


+ gcloud storage ls gs://sbox-johnny-001-mldemo/models/
gs://sbox-johnny-001-mldemo/models/image_classifier.h5


## 2. Deploy Cloud Run from `serving/`

Rebuilds the container (bakes in `serving/models/image_classifier.h5`). First build is slow; idle cost is ~$0.

In [4]:
# Dockerfile expects the model at serving/models/image_classifier.h5
os.makedirs("serving/models", exist_ok=True)
sh(f"cp {MODEL_LOCAL} serving/models/image_classifier.h5")

# --source=serving: Cloud Build builds serving/Dockerfile and deploys the image
# --min-instances=0: scale to zero when idle (no always-on charge)
# --allow-unauthenticated: URL is publicly callable (lock down later if needed)
# First deploy is slow (TensorFlow image); later deploys are usually faster
sh(
    f"gcloud run deploy {SERVICE} "
    f"--source=serving "
    f"--region={REGION} "
    f"--allow-unauthenticated "
    f"--min-instances=0 --max-instances=2 "
    f"--memory=1Gi --cpu=1 --concurrency=1 --timeout=60 --quiet"
)

# Public HTTPS API base URL (not a website UI — use /health and /predict)
url = subprocess.check_output(
    f"gcloud run services describe {SERVICE} --region={REGION} --format='value(status.url)'",
    shell=True,
    text=True,
).strip()
print("URL:", url)

+ cp models/image_classifier.h5 serving/models/image_classifier.h5
+ gcloud run deploy cat-dog-classifier --source=serving --region=us-central1 --allow-unauthenticated --min-instances=0 --max-instances=2 --memory=1Gi --cpu=1 --concurrency=1 --timeout=60 --quiet


Building using Dockerfile and deploying container to Cloud Run service [cat-dog-classifier] in project [sbox-johnny-001] region [us-central1]
Building and deploying...
Validating configuration.......done
Uploading sources.....................................................................................................................................................................................................................done
Building Container...............................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

URL: https://cat-dog-classifier-qdu3axmquq-uc.a.run.app


## 3. Call the service

In [5]:
import requests

# Look up the live Cloud Run URL (same as after deploy)
url = subprocess.check_output(
    f"gcloud run services describe {SERVICE} --region={REGION} --format='value(status.url)'",
    shell=True,
    text=True,
).strip()

# After idle, cold start can take a while — timeout is intentionally high
print(requests.get(f"{url}/health", timeout=120).json())

# POST multipart field must be named "file" (see serving/main.py)
# Response is JSON: {"label": "cat"|"dog", "confidence": ..., "raw_score": ...}
for path in ("dog_test.jpg", "cat_test.jpg"):
    with open(path, "rb") as f:
        print(path, requests.post(
            f"{url}/predict",
            files={"file": (path, f, "image/jpeg")},
            timeout=120,
        ).json())

{'status': 'ok', 'model_loaded': True}
dog_test.jpg {'label': 'dog', 'confidence': 0.7219, 'raw_score': 0.7219}
cat_test.jpg {'label': 'cat', 'confidence': 0.9981, 'raw_score': 0.0019}


4. Tear down (optional)

Only if you want zero Cloud Run residual. GCS storage is cheap; delete the bucket only if you are done.

In [ ]:
# Uncomment to delete:
# sh(f"gcloud run services delete {SERVICE} --region={REGION} --quiet")
# sh(f"gcloud storage rm -r {BUCKET}")
print("left commented on purpose")